<a href="https://colab.research.google.com/github/tomasroel/DMeyf_Competencias/blob/main/modelos/2_0_competencia1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LightGBM  Optimizacion Bayesiana

Tarea para el Hogar 04,  versión Gustavo Pasquini: 2026

## 📝 Notebook 2.0 (Tomás)

Parte del notebook 1.0 (el de la cátedra, con rutas a Drive y lectura única del dataset). Cambios de esta versión:

* **Data drifting (3.3)**: PSI de cada variable, original o `new_`, entre 202104 y los demás meses, con una tabla ordenada de las que más cambiaron y una sugerencia por umbral.
* **Features nuevas (3.2)**: las 35 features `new_` que se armaron para el análisis de clustering.
* **Lags y delta lags (3.4)**: `_lag1` y `_delta1` de todas las variables, originales y nuevas.
* **Training con dos meses**: 202103 y 202104, con test en 202106.
* **Prueba rápida (4.3)**: antes de la BO, modelos con hiperparámetros fijos y 5 semillas, y su ganancia promedio en 202106, para saber rápido si los cambios funcionan.
* **Ganancia máxima y curva**: en la prueba rápida y al final, ganancia máxima, cantidad de envíos y gráfico de líneas.
* **Importancia de variables**: en la prueba rápida y al final, gráfico de las 30 variables más importantes y peso de cada tipo (original, nueva, lag, delta).
* **Variables predictoras (4.2)**: `foto_mes` deja de ser predictora y las variables excluidas por drifting se eligen en `PARAM$drift$excluir`.

El código de modelado (partición, undersampling, BO, entrenamiento final, ganancia) no cambió.

**Estructura**

| Sección | Contenido |
|---|---|
| 1.0 | Seteo del ambiente (Python) |
| 2.0 | Inicio, librerías, parámetros y funciones |
| 3.0 | Dataset: lectura, clase, features nuevas, drifting, lags |
| 4.0 | Training, prueba rápida y optimización bayesiana |
| 5.0 | Producción: modelo final y ganancia en 202106 |

**Dónde queda cada cosa**

| Qué | Dónde | ¿Persiste? |
|---|---|---|
| `competencia_01_crudo.csv` | Drive: `competencia1/datasets/` | Sí |
| Copia de trabajo del crudo | Colab: `/content/datasets/` | No |
| `drift_psi.txt` | Drive: `competencia1/outputs/<experimento>/` | Sí |
| `BO_log.txt` | Drive: `competencia1/outputs/<experimento>/` | Sí |
| `importancia_prueba.txt`, `importancia_final.txt` | Drive: `competencia1/outputs/<experimento>/` | Sí |
| `pruebas_rapidas_semillas.txt` | Drive: `competencia1/outputs/` | Sí |

**Cómo correrlo**

1. Runtime en **Python 3**: correr la sección 1.0 (monta Drive y baja el dataset).
2. Cambiar el runtime a **R** y correr el resto, de arriba hacia abajo.
3. Después de 3.3, mirar la tabla de drifting y editar `PARAM$drift$excluir` antes de seguir.
4. Después de 4.3, mirar la ganancia de la prueba rápida antes de lanzar la BO.

Las celdas markdown que empiezan con 📝 y los comentarios `# >>` son explicaciones agregadas.

## 1.0  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

📝 **Montaje de Drive.** Conecta la máquina virtual de Colab con tu Google Drive en `/content/.drive`. Sin esto, todo lo que se grabe se pierde al cerrar la sesión.

In [ ]:
# primero establecer el Runtime de Python 3
# >> Monta tu Google Drive en /content/.drive (pide autorizacion la primera vez).
# >> "Mi unidad" queda accesible como /content/.drive/My Drive
from google.colab import drive
drive.mount('/content/.drive')

📝 **Carpetas y descarga del dataset.**

* Crea `competencia1/` en Drive (si no existe) y un link simbólico `/content/buckets/b1` que apunta ahí. A partir de acá, `/content/buckets/b1/...` **es** tu carpeta de Drive.
* Crea `outputs/` y `datasets/` dentro de `competencia1/`, y `/content/datasets` en el disco local.
* `descargar()` baja el crudo del bucket de la materia a Drive solo si no está, y después lo copia al disco local, que es de donde lo lee R (más rápido que leer de Drive).

In [ ]:
%%shell

# >> CAMBIO: carpeta base en Drive = Mi unidad/competencias_dmeyf/competencia1
# >> (original: "/content/.drive/My Drive/dmeyf")
mkdir -p "/content/.drive/My Drive/competencias_dmeyf/competencia1"
mkdir -p "/content/buckets"
# >> link simbolico: /content/buckets/b1 apunta a la carpeta de Drive,
# >> asi el codigo R usa una ruta corta y sin espacios
ln -sfn  "/content/.drive/My Drive/competencias_dmeyf/competencia1" /content/buckets/b1


# >> CAMBIO: los resultados van a competencia1/outputs (original: b1/exp)
mkdir -p /content/buckets/b1/outputs
# >> competencia1/datasets en Drive: copia persistente del dataset crudo
mkdir -p /content/buckets/b1/datasets
# >> /content/datasets: disco local de Colab (rapido, pero se borra al cerrar la sesion)
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/dmeyf2026-9c6f/"
  archivo="$1"

  # >> paso 1: si el archivo NO esta en Drive, lo baja del bucket de la materia
  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  # >> paso 2: si NO esta en el disco local, lo copia desde Drive
  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "competencia_01_crudo.csv"


## 2.0 Inicio, librerías y parámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.1 Inicio

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

📝 Limpia la memoria de R antes de empezar. El dataset todavía no está cargado: se lee más abajo, en 3.1.

In [ ]:
# limpio la memoria
# >> borra todos los objetos que haya en memoria antes de empezar
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

### 2.2 Carga de Librerias

Esta parte lleva  demasiados   minutos la primera vez en Google Colab
<br> por suerte a partir del 15-septiembre empezaremos a trabajar con Google Cloud, en donde todas las librerias, de Python R y Julia ya estarán instaladas

In [ ]:
# instalo paquetes
# >> R.utils: utilidades (lectura de .gz) / rlist: manejo de listas (list.names en loguear)
install.packages( c("R.utils", "rlist" ) )

In [ ]:
# Librerias  modernas  para la Bayesian Optimization
# >> bbotk: framework de optimizacion black-box / mlr3mbo: optimizacion bayesiana
# >> DiceKriging: el Gaussian Process (kriging) que usa la BO como modelo surrogate
install.packages( c("bbotk", "mlr3", "mlr3mbo", "mlr3learners", "DiceKriging") )

In [ ]:
# el LightGBM
install.packages(c("lightgbm"))

In [ ]:
library(data.table)
library(R.utils)
library(rlist)
library(bbotk)
library(mlr3)
library(mlr3mbo)
library(mlr3learners)
# library(paradox)
library(DiceKriging)
library(lightgbm)

### 2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

📝 **Parámetros del experimento.** Todo lo configurable vive en la lista `PARAM`. `PARAM$experimento` define el nombre de la subcarpeta de salida en `competencia1/outputs/`.

In [ ]:
PARAM <- list()
# >> nombre del experimento = nombre de la subcarpeta en competencia1/outputs/
# >> cambiarlo en cada corrida nueva (si no, el BO_log.txt se acumula sobre el anterior)
PARAM$experimento <- "fe_020"
# >> semilla usada para la particion, el undersampling y LightGBM
PARAM$semilla_primigenia <-  911341   # La semilla de Laura Pasquini

📝 **Esquema temporal.** La BO y el modelo final usan 202103 y 202104 (los dos meses con clase completa si estamos "parados" en 202106). El modelo se aplica sobre 202106 y, como ahí la clase es conocida, se puede medir la ganancia real para distintos cortes de envíos.

In [ ]:
# training y future
# >> 70% de los meses de train a training, 30% a validation
PARAM$training_pct <- 70
# >> meses usados en la optimizacion bayesiana
PARAM$train <- c(202103, 202104)
# >> meses usados para entrenar el modelo final
PARAM$train_final <- c(202103, 202104)
# >> mes donde se aplica el modelo y se mide la ganancia (tiene clase conocida)
PARAM$future <- c(202106)
# >> cantidades de envios a evaluar: 9000, 9500, ..., 15000
PARAM$cortes <- seq(9000, 15000, by= 500)

📝 **Undersampling.** Para entrenar se usan todos los BAJA y solo el 10% de los CONTINUA. Acelera mucho el entrenamiento, a costa de que las probabilidades que predice el modelo dejan de estar en la escala real.

In [ ]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.1

📝 **Iteraciones de la BO.** Con 3 hiperparámetros, `mlr3mbo` arranca con un diseño inicial de 4 × 3 = 12 puntos. Con tope 20 quedan 8 evaluaciones más, propuestas de a una por el modelo surrogate.

In [ ]:
# >> tope de evaluaciones de la BO (ver nota en la celda markdown de arriba)
PARAM$hyperparametertuning$iteraciones <- 20 # iteraciones bayesianas

📝 **Parámetros fijos de LightGBM.** Son los que no se optimizan. Lo central de la estrategia:

* `learning_rate = 0.005` muy bajo y fijo, compensado con muchas iteraciones.
* `feature_fraction = 0.5` y `max_bin = 31` como regularización.
* `min_data_in_leaf = 0`: el tamaño mínimo de hoja se controla solo con `min_sum_hessian_in_leaf`.
* Sin early stopping y sin rebalanceo interno (`is_unbalance = FALSE`, `scale_pos_weight = 1`).
* Los tres últimos (`num_iterations`, `num_leaves`, `min_sum_hessian_in_leaf`) son los que pisa la BO.

In [ ]:
# Parametros LightGBM

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  # >> gradient boosting clasico
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  # >> clasificacion binaria (clase01), minimiza logloss
  objective= "binary",
  # >> metrica que se calcula en validation: area bajo la curva precision-recall
  metric= "average_precision",
  feature_pre_filter= FALSE,
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  force_row_wise= TRUE,
  deterministic= TRUE,
  verbosity= -100,

  # >> semilla interna de LightGBM
  seed= PARAM$semilla_primigenia,

  # >> sin limite de profundidad; la complejidad se controla con num_leaves
  max_depth= -1L,
  # >> sin ganancia minima para abrir un split y sin regularizacion L1 ni L2
  min_gain_to_split= 0,
  lambda_l1= 0.0,
  lambda_l2= 0.0,
  # >> cada variable numerica se discretiza en 31 bins (mas rapido, regulariza)
  max_bin= 31L,

  # >> sin bagging de filas: cada arbol ve todas las filas del training
  bagging_fraction= 1.0,
  pos_bagging_fraction= 1.0,
  neg_bagging_fraction= 1.0,
  # >> NO se rebalancea dentro de LightGBM (el desbalance se trata con el undersampling)
  is_unbalance= FALSE,
  scale_pos_weight= 1.0,

  # >> drop_rate / max_drop / skip_drop solo aplican a boosting=dart; aca no tienen efecto
  drop_rate= 0.1,
  max_drop= 50,
  skip_drop= 0.5,

  extra_trees= FALSE,

  # >> 0 = sin early stopping: siempre se entrenan las num_iterations completas
  early_stopping= 0,
  # >> sin minimo de registros por hoja; el tamaño minimo de hoja se regula
  # >> con min_sum_hessian_in_leaf, que es el que optimiza la BO
  min_data_in_leaf= 0,     # fijo, esto es FUNDAMENTAL
  # >> cada arbol usa el 50% de las variables, elegidas al azar
  feature_fraction= 0.50,  # fijo
  # >> learning rate muy bajo y fijo; se compensa con muchas iteraciones
  learning_rate= 0.005,    # fijo

  # >> los tres de abajo son placeholders: los pisa cada iteracion de la BO
  num_iterations= 20000,     # se modificara
  num_leaves= 1024,          # se modificara
  min_sum_hessian_in_leaf= 1e-06  # se modificara
)


Aqui se definen los hiperparámetros de LightGBM que participan de la Bayesian Optimization

📝 **Espacio de búsqueda.** Rangos de los 3 hiperparámetros que explora la BO. `ps()`, `p_int()` y `p_dbl()` son de la librería `paradox`.

In [ ]:
# Aqui se cargan los bordes de los hiperparametros de la BO
#  es vital entender que se optimiza min_sum_hessian_in_leaf y que  min_data_in_leaf se deja fijo en CERO

PARAM$hypeparametertuning$hs <- ps(
  # >> cantidad de arboles: entero entre 64 y 4096
  num_iterations= p_int(lower= 64L,  upper= 4096L),
  # >> hojas maximas por arbol: entero entre 16 y 2048
  num_leaves= p_int(lower= 16L,  upper= 2048L),
  # >> suma minima de hessianos por hoja: real entre 1e-06 y 0.1
  # >> (en logloss el hessiano de una fila es p*(1-p): funciona como un 'conteo blando' de registros)
  min_sum_hessian_in_leaf= p_dbl( lower=1e-06, upper=0.1)
)

📝 **Parámetros de drifting y columnas no predictoras.** Definen qué meses se comparan en el análisis de 3.3, contra qué mes de referencia y con qué umbrales se arma la sugerencia. `PARAM$campos_no_predictores` son columnas que nunca entran al modelo: con dos meses de training, `foto_mes` deja de ser constante y el modelo podría cortar por mes.

In [ ]:
# analisis de data drifting (PSI)
PARAM$drift$mes_referencia <- 202104     # mes contra el que se comparan los demas
PARAM$drift$meses_historia <- c(202103, 202104, 202105, 202106)
PARAM$drift$mes_aplicacion <- 202108     # mes del submit final
PARAM$drift$bins <- 10                   # bins para los valores distintos de cero
PARAM$drift$umbral_revisar <- 0.10       # PSI desde el cual conviene mirar la variable
PARAM$drift$umbral_fuerte <- 0.25        # PSI desde el cual el cambio es fuerte

# columnas que nunca entran como predictoras
PARAM$campos_no_predictores <- c("foto_mes")

📝 **Parámetros de la prueba rápida.** Antes de la BO se entrena un modelo por cada semilla de `PARAM$prueba$semillas`, con hiperparámetros fijos, y se mide la ganancia promedio en 202106 (sección 4.3). Los hiperparámetros se toman del mejor resultado del experimento indicado en `PARAM$prueba$experimento_hp`; si ese experimento no tiene `BO_log.txt`, se usan los manuales, que son los mejores de la corrida de la cátedra.

In [ ]:
# prueba rapida previa a la BO
#  semillas de LightGBM: se entrena un modelo por semilla y se promedia la ganancia
#  reemplazar las cuatro ultimas por las propias
PARAM$prueba$semillas <- c(PARAM$semilla_primigenia, 100003, 200003, 300007, 400009)

#  experimento del que se toman los mejores hiperparametros (lee su BO_log.txt)
PARAM$prueba$experimento_hp <- "base_010"

#  hiperparametros que se usan si ese BO_log.txt no existe
PARAM$prueba$hp_manual <- list(
  num_iterations= 2848,
  num_leaves= 71,
  min_sum_hessian_in_leaf= 0.0067
)

#  archivo donde se acumula una fila por cada prueba rapida, de todos los experimentos
PARAM$prueba$archivo_log <- "/content/buckets/b1/outputs/pruebas_rapidas_semillas.txt"

### 2.4 Funciones

📝 **Función `particionar()`.** Agrega al dataset una columna `fold` con una partición al azar y estratificada (acá 70/30 por `clase_ternaria`).

In [ ]:
# particionar agrega una columna llamada fold a un dataset
#   que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30),
#  agrupa=clase_ternaria, seed=semilla)   crea una particion 70, 30

particionar <- function(data, division, agrupa= "", campo= "fold", start= 1, seed= NA) {
  # >> fija la semilla para que la particion sea reproducible
  if (!is.na(seed)) set.seed(seed, "L'Ecuyer-CMRG")

  # >> arma un bloque con 70 unos y 30 doses (segun `division`)
  bloque <- unlist(mapply(
    function(x, y) {rep(y, x)},division, seq(from= start, length.out= length(division))))

  # >> dentro de cada clase (by= agrupa) repite el bloque, lo mezcla y asigna fold 1 o 2:
  # >> particion estratificada, cada clase queda 70/30
  data[, (campo) := sample(rep(bloque,ceiling(.N / length(bloque))))[1:.N],by= agrupa]
}

📝 **Función `loguear()`.** Agrega una fila a un archivo de texto separado por tabs, con fecha-hora y todos los valores de la lista que recibe. Así se arma `BO_log.txt`, una fila por cada modelo evaluado en la BO.

In [ ]:
# logueo al archivo BO_log.txt
loguear  <- function( preg, arch=NA, verbose=TRUE )
{
  t0 <- Sys.time()
  reg <- copy( preg )
  # >> (no aplica en este notebook: param_fijos no tiene feature_contri)
  if( "feature_contri" %in%  names(reg) ) {
    reg$feature_contri <- reg$feature_contri[1]
  }
  archivo <- arch
  if( is.na(arch) ) archivo <- paste0( folder, substitute( reg), ext )


  # >> si el archivo no existe, primero escribe la fila de titulos
  if( !file.exists( archivo ) )
  {
    # Escribo los titulos
    linea  <- paste0( "fecha\t",
                      paste( list.names(reg), collapse="\t" ), "\n" )

    cat( linea, file=archivo )
  }

  # escribo el registro
  # >> una fila por evaluacion: fecha-hora + todos los valores, separados por tab
  linea  <- paste0( format(t0, "%Y%m%d.%H%M%S"),  "\t",     # la fecha y hora
                    gsub( ", ", "\t", toString( reg ) ),  "\n" )

  cat( linea, file=archivo, append=TRUE )  # grabo al archivo

  if( verbose )  cat( linea )   # imprimo por pantalla
}

📝 **Funciones de ganancia.** `curva_ganancia()` ordena los clientes por probabilidad y acumula la ganancia. `graficar_ganancia()` imprime la ganancia máxima con su cantidad de envíos y dibuja la curva; si recibe las curvas de varias semillas, las dibuja en gris detrás del promedio. Se usan en la prueba rápida (4.3) y al final (5.0).

In [ ]:
# arma la curva de ganancia: ordena por probabilidad descendente y acumula
#  devuelve una tabla con prob, gan, ganancia_acumulada y pos (cantidad de envios)
curva_ganancia <- function(prob, clase_ternaria) {
  tb <- data.table(prob= prob, clase_ternaria= clase_ternaria)
  tb[, gan := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]
  setorder(tb, -prob)
  tb[, ganancia_acumulada := cumsum(gan)]
  tb[, pos := sequence(.N)]
  tb[]
}

# imprime la ganancia maxima con su cantidad de envios y dibuja la curva
#  tb: tabla ordenada por cantidad de envios, con columnas pos y ganancia_acumulada
#  curvas_semillas: opcional, matriz con una curva por columna (una por semilla);
#   en ese caso tb es la curva promedio y las curvas individuales se dibujan en gris
#  devuelve (de forma invisible) la fila del maximo
graficar_ganancia <- function(tb, titulo, curvas_semillas= NULL) {
  mejor <- tb[which.max(ganancia_acumulada)]

  etiqueta <- "Ganancia maxima:"
  if (!is.null(curvas_semillas)) {
    etiqueta <- paste0("Ganancia maxima (promedio de ", ncol(curvas_semillas), " semillas):")
  }
  cat(etiqueta, format(round(mejor$ganancia_acumulada), big.mark= ".", decimal.mark= ","), "\n")
  cat("Envios:", format(mejor$pos, big.mark= ".", decimal.mark= ","), "\n")

  options(repr.plot.width= 12, repr.plot.height= 6, scipen= 999)

  # muestro hasta el doble del corte mas grande, o un poco mas alla del maximo si cae afuera
  tope <- min(nrow(tb), max(2 * max(PARAM$cortes), ceiling(1.2 * mejor$pos)))
  curva <- tb[pos <= tope]

  rango <- range(curva$ganancia_acumulada)
  if (!is.null(curvas_semillas)) rango <- range(rango, curvas_semillas[seq_len(tope), ])

  plot(curva$pos, curva$ganancia_acumulada / 1e6,
    type= "n", ylim= rango / 1e6,
    xlab= "Cantidad de envios", ylab= "Ganancia (millones)", main= titulo
  )
  grid()

  # una linea gris por semilla y la curva principal encima
  if (!is.null(curvas_semillas)) {
    for (j in seq_len(ncol(curvas_semillas))) {
      lines(seq_len(tope), curvas_semillas[seq_len(tope), j] / 1e6, col= "grey70")
    }
    legend("bottomright", legend= c("promedio", "cada semilla"),
      col= c("steelblue", "grey70"), lwd= c(2, 1), bty= "n")
  }
  lines(curva$pos, curva$ganancia_acumulada / 1e6, lwd= 2, col= "steelblue")

  # marco el maximo
  abline(v= mejor$pos, lty= 2, col= "red")
  points(mejor$pos, mejor$ganancia_acumulada / 1e6, pch= 19, col= "red")
  text(mejor$pos, mejor$ganancia_acumulada / 1e6, pos= 4, col= "red",
    labels= paste0(format(round(mejor$ganancia_acumulada / 1e6, 1), decimal.mark= ","), " M con ",
      format(mejor$pos, big.mark= ".", decimal.mark= ","), " envios"))

  invisible(mejor)
}

📝 **Funciones de importancia de variables.** `resumir_importancia()` arma la tabla de importancia (promediando si hay varios modelos) y clasifica cada variable como original, nueva, lag o delta. `graficar_importancia()` dibuja las 30 más importantes y muestra cuánto pesa cada tipo.

In [ ]:
# arma la tabla de importancia a partir de una lista de salidas de lgb.importance()
#  con varios modelos (uno por semilla) promedia la importancia
#  gain: proporcion de la mejora total del modelo que aporta cada variable
#  tipo: original, nueva (new_), lag (_lag1) o delta (_delta1)
resumir_importancia <- function(lista) {
  tb <- rbindlist(lista)
  tb <- tb[, list(gain= sum(Gain) / length(lista)), by= list(variable= Feature)]
  tb[, tipo := fcase(
    grepl("_lag1$", variable), "lag",
    grepl("_delta1$", variable), "delta",
    grepl("^new_", variable), "nueva",
    default= "original"
  )]
  setorder(tb, -gain)
  tb[, puesto := .I]
  tb[, gain_acumulado := cumsum(gain)]
  tb[]
}

# grafico de barras con las variables mas importantes, coloreadas por tipo,
#  y tabla con el peso de cada tipo de variable
graficar_importancia <- function(tb, titulo, top= 30) {
  colores <- c(original= "steelblue", nueva= "darkorange", lag= "seagreen", delta= "mediumpurple")

  cat("Variables usadas por el modelo:", nrow(tb), "de", length(campos_buenos), "\n")
  print(tb[, list(variables= .N, gain_pct= round(100 * sum(gain), 1)), keyby= tipo])

  # las barras se dibujan de abajo hacia arriba: invierto para que la primera quede arriba
  mostrar <- tb[seq_len(min(top, .N))][.N:1]

  options(repr.plot.width= 12, repr.plot.height= 9)
  margenes <- par(mar= c(4, 17, 3, 1))
  barplot(100 * mostrar$gain,
    names.arg= mostrar$variable, horiz= TRUE, las= 1, cex.names= 0.8,
    col= colores[mostrar$tipo], border= NA,
    xlab= "Importancia (% del gain total)", main= titulo
  )
  legend("bottomright", legend= names(colores), fill= colores, border= NA, bty= "n")
  par(margenes)
}

## 3.0 Dataset

### 3.1 Lectura y clase_ternaria

📝 **Carpeta de trabajo.** Se para en `competencia1/outputs/<experimento>/` (en Drive). Todo archivo que se grabe con ruta relativa, como `BO_log.txt`, queda ahí.

In [ ]:
# carpeta de trabajo

# >> CAMBIO: carpeta de salida en Drive = competencia1/outputs (original: "/content/buckets/b1/exp")
setwd("/content/buckets/b1/outputs")
# >> crea la subcarpeta del experimento (outputs/<experimento>) y se para ahi
dir.create(PARAM$experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/outputs/", PARAM$experimento))

📝 **Lectura del dataset y cálculo de `clase_ternaria`.** El crudo se lee una sola vez, se le calcula la clase y `dataset` queda en memoria para el resto del notebook.

El crudo no trae la clase: se deduce de si el cliente aparece o no en los meses siguientes.

* **BAJA+1**: no está en la foto del mes siguiente.
* **BAJA+2**: está en el mes siguiente, pero no en el subsiguiente.
* **CONTINUA**: está en los dos meses siguientes.

Como hacen falta dos meses de futuro, la clase solo queda completa hasta 202106. En 202107 solo se pueden marcar los BAJA+1 (el resto queda `NA`) y 202108 queda todo en `NA`.

In [ ]:
# leo el dataset
# >> CAMBIO: unica lectura del dataset en todo el notebook (crudo, sin clase_ternaria)
# >> se agrega stringsAsFactors= TRUE, que era la opcion de la lectura que usaba el modelo
dataset <- fread("/content/datasets/competencia_01_crudo.csv", stringsAsFactors= TRUE )

# calculo el periodo0 consecutivo
# >> tabla auxiliar minima: posicion original de la fila, cliente y periodo0
# >> periodo0 = año*12 + mes -> numera los meses de corrido (202112 y 202201 quedan consecutivos)
dsimple <- dataset[, list(
    "pos" = .I,
    numero_de_cliente,
    periodo0 = as.integer(foto_mes/100)*12 +  foto_mes%%100 ) ]


# ordeno
setorder( dsimple, numero_de_cliente, periodo0 )

# calculo topes
# >> ultimo mes del dataset (202108) y anteultimo (202107)
periodo_ultimo <- dsimple[, max(periodo0) ]
periodo_anteultimo <- periodo_ultimo - 1


# calculo los leads de orden 1 y 2
# >> para cada cliente: periodo1 = el proximo mes en que aparece, periodo2 = el siguiente
# >> (NA si el cliente no vuelve a aparecer)
dsimple[, c("periodo1", "periodo2") :=
    shift(periodo0, n=1:2, fill=NA, type="lead"),  numero_de_cliente ]

# assign most common class values = "CONTINUA"
# >> por defecto CONTINUA, solo para meses con 2 meses de futuro observable (hasta 202106)
dsimple[ periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA" ]

# calculo BAJA+1
# >> BAJA+1: el cliente NO esta en el mes siguiente (no hay registro o hay un hueco)
# >> se puede calcular hasta 202107 (alcanza con 1 mes de futuro)
dsimple[ periodo0 < periodo_ultimo &
    ( is.na(periodo1) | periodo0 + 1 < periodo1 ),
    clase_ternaria := "BAJA+1" ]

# calculo BAJA+2
# >> BAJA+2: esta en el mes siguiente pero NO en el subsiguiente
# >> se puede calcular hasta 202106; 202107 queda con BAJA+1 o NA y 202108 todo NA
dsimple[ periodo0 < periodo_anteultimo & (periodo0+1 == periodo1 )
    & ( is.na(periodo2) | periodo0 + 2 < periodo2 ),
    clase_ternaria := "BAJA+2" ]


# pego el resultado en el dataset original
# >> vuelve al orden original de filas para pegar la clase en el dataset completo
setorder( dsimple, pos )
dataset[, clase_ternaria := dsimple$clase_ternaria ]

# >> CAMBIO: ya no se graba ni se relee competencia_01.csv.gz;
# >> `dataset` queda en memoria, con clase_ternaria, para el resto del notebook
# >> libero la tabla auxiliar
rm( dsimple )

📝 **Control.** Tabla de cantidad de clientes por mes y clase. 202103 tiene 960 BAJA+2 y 1.019 BAJA+1; 202104 tiene 1.139 BAJA+2 y 964 BAJA+1; 202106 tiene 1.098 BAJA+2.

In [ ]:
# >> CAMBIO: keyby ordena solo la tabla de conteos y NO reordena `dataset`
# >> (original: setorder( dataset, foto_mes, clase_ternaria, numero_de_cliente) )
dataset[, .N, keyby= list(foto_mes, clase_ternaria)]

### 3.2 Features nuevas

📝 **Features nuevas.** Las 35 features que se armaron para el análisis de clustering: 31 sumas de variables del mismo tipo (por ejemplo Visa + Master), 2 restas y 2 derivadas de otras nuevas. Se calculan antes del análisis de drifting y de los lags, para que entren en los dos. Si falta alguna columna de entrada, la celda frena y dice cuál.

In [ ]:
# features que son sumas de variables originales
sumas <- list(
  new_total_margen                 = c("mactivos_margen", "mpasivos_margen"),
  new_monto_cc                     = c("mcuenta_corriente", "mcuenta_corriente_adicional"),
  new_cant_cuentas                 = c("ccaja_ahorro", "ccuenta_corriente"),
  new_cant_consumos_tarjcredito    = c("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones"),
  new_monto_consumos_tarjcredito   = c("mtarjeta_visa_consumo", "mtarjeta_master_consumo"),
  new_cant_prestamos               = c("cprestamos_personales", "cprestamos_prendarios", "cprestamos_hipotecarios"),
  new_monto_prestamos              = c("mprestamos_personales", "mprestamos_prendarios", "mprestamos_hipotecarios"),
  new_monto_pf                     = c("mplazo_fijo_dolares", "mplazo_fijo_pesos"),
  new_monto_inv                    = c("minversion1_pesos", "minversion1_dolares", "minversion2"),
  new_cant_inv_total               = c("cplazo_fijo", "cinversion1", "cinversion2"),
  new_cant_seguros                 = c("cseguro_vida", "cseguro_auto", "cseguro_vivienda", "cseguro_accidentes_personales"),
  new_total_payroll                = c("mpayroll", "mpayroll2"),
  new_cant_debitos_aut             = c("ccuenta_debitos_automaticos", "ctarjeta_visa_debitos_automaticos", "ctarjeta_master_debitos_automaticos"),
  new_monto_debitos_aut            = c("mcuenta_debitos_automaticos", "mttarjeta_visa_debitos_automaticos", "mttarjeta_master_debitos_automaticos"),
  new_cant_pagos_servicios         = c("cpagodeservicios", "cpagomiscuentas"),
  new_monto_pagos_servicios        = c("mpagodeservicios", "mpagomiscuentas"),
  new_cant_descuentos_beneficios   = c("ccajeros_propios_descuentos", "ctarjeta_visa_descuentos", "ctarjeta_master_descuentos"),
  new_monto_descuentos_beneficios  = c("mcajeros_propios_descuentos", "mtarjeta_visa_descuentos", "mtarjeta_master_descuentos"),
  new_monto_comisiones_mant        = c("mcomisiones_mantenimiento", "mcomisiones_otras"),
  new_egresos_totales              = c("mtransferencias_emitidas", "mcheques_emitidos", "mextraccion_autoservicio"),
  new_actividad_cajas              = c("ccajas_transacciones", "ccajas_depositos", "ccajas_extracciones", "ccajas_otras"),
  new_actividad_atm                = c("catm_trx", "catm_trx_other"),
  new_monto_atm                    = c("matm", "matm_other"),
  new_delinquency_cards            = c("Master_delinquency", "Visa_delinquency"),
  new_limite_financiacion_tarjetas = c("Master_mfinanciacion_limite", "Visa_mfinanciacion_limite"),
  new_saldo_tarjetas               = c("Master_msaldototal", "Visa_msaldototal"),
  new_consumos_tarjetas            = c("Master_mconsumospesos", "Master_mconsumosdolares", "Visa_mconsumospesos", "Visa_mconsumosdolares"),
  new_limite_compras_combinado     = c("Master_mlimitecompra", "Visa_mlimitecompra"),
  new_adelantos_tarjetas           = c("Master_madelantopesos", "Master_madelantodolares", "Visa_madelantopesos", "Visa_madelantodolares"),
  new_pagos_combinado              = c("Master_mpagado", "Visa_mpagado"),
  new_cant_consumos_tarjetas       = c("Master_cconsumos", "Visa_cconsumos")
)

# features que no son sumas: se anotan las variables originales que usan
otras <- list(
  new_msaldo_pesos         = c("mcuentas_saldo", "mcaja_ahorro_dolares"),
  new_diff_buy_sell        = c("mforex_buy", "mforex_sell"),
  new_ingresos_totales     = c("mtransferencias_recibidas", "mpayroll", "mpayroll2", "mcheques_depositados"),
  new_share_margen_activos = c("mactivos_margen", "mpasivos_margen")
)

# de que variables originales depende cada feature nueva
insumos_new <- c(sumas, otras)

# chequeo previo: que existan todas las columnas de entrada
faltantes <- setdiff(unique(unlist(insumos_new)), colnames(dataset))
if (length(faltantes) > 0) stop("Faltan columnas en el dataset: ", paste(faltantes, collapse= ", "))

# sumas: un NA se toma como cero (en general significa que el cliente no tiene ese producto)
for (nueva in names(sumas)) {
  dataset[, (nueva) := rowSums(.SD, na.rm= TRUE), .SDcols= sumas[[nueva]]]
}

# restas
dataset[, new_msaldo_pesos := mcuentas_saldo - mcaja_ahorro_dolares]
dataset[, new_diff_buy_sell := mforex_buy - mforex_sell]

# derivadas de otras nuevas
dataset[, new_ingresos_totales := rowSums(.SD, na.rm= TRUE),
  .SDcols= c("mtransferencias_recibidas", "new_total_payroll", "mcheques_depositados")]

dataset[, new_share_margen_activos := fifelse(new_total_margen == 0, NA_real_,
  mactivos_margen / new_total_margen)]

cat("Features nuevas:", length(insumos_new), "\n")
cat("Columnas del dataset:", ncol(dataset), "\n")

### 3.3 Data drifting

📝 **Análisis de data drifting.** Compara la distribución de cada variable en el mes de referencia (202104) contra cada uno de los otros meses: 202103, 202105 y 202106 por un lado, y 202108, el mes del submit final, por otro.

La medida es el **PSI** (Population Stability Index). Para cada variable se arman bins: uno para los `NA`, uno para los ceros y hasta 10 para el resto, cortados por los deciles de los valores no nulos del mes de referencia. El PSI compara qué proporción de clientes cae en cada bin en un mes y en el otro. Vale 0 si las dos distribuciones son iguales y crece cuanto más se diferencian.

Umbrales habituales (son una convención, no una regla):

| PSI | Lectura |
|---|---|
| menos de 0,10 | estable |
| 0,10 a 0,25 | conviene revisar |
| más de 0,25 | cambio fuerte |

El análisis incluye las variables originales y las features `new_`. Se hace antes de crear los lags, así que los `_lag1` y `_delta1` no se analizan.

#### Salsa Magica (dedicada a Laura Pasquini)

Los EDAs que estan haciendo los alumnos de los lunes son pura falopa, y de la mala. El uso de Agentes de la IA solo les empera el daño cerebral, generan pedf larguisimos, que no te dan ganas de leerlos, carentes de insights y alma.
<br>El unico EDA que tiene realmente sentido, en este caso, es comparar la distribución de cada variable del mes donde se entrena 202104 y el mes donde se va a aplicar el modelo 202106, buscando Data Drifting.
<br>Aun con los reales datos del futuro, donde no se tiene la clase, SIEMPRE se puede analizar el Data Drifting.
<br>Para las variables con distribuciones distintas se puede hacer:
* Transformar la variable para hacerla homogenea entre los dos periodos
* Eliminarla (es lo simple, que se eligió hacer en este caso)

In [ ]:
# PSI (Population Stability Index) entre el mes de referencia y otro mes
#  bins: uno para los NA, uno para los ceros y hasta `bins` para el resto,
#  cortados por los cuantiles de los valores no nulos del mes de referencia
psi <- function(x_ref, x_new, bins= 10) {
  probs <- seq(0, 1, length.out= bins + 1)[-c(1, bins + 1)]
  cortes <- unique(quantile(x_ref[x_ref != 0], probs= probs, na.rm= TRUE, names= FALSE))
  cortes <- cortes[!is.na(cortes)]

  asignar_bin <- function(x) {
    b <- findInterval(x, cortes) + 3L   # 3, 4, ... valores distintos de cero
    b[x == 0] <- 2L                     # 2 = ceros
    b[is.na(x)] <- 1L                   # 1 = NA
    tabulate(b, nbins= length(cortes) + 3L) / length(x)
  }

  # piso para no dividir por cero ni tomar log(0)
  p_ref <- pmax(asignar_bin(x_ref), 1e-4)
  p_new <- pmax(asignar_bin(x_new), 1e-4)

  sum((p_new - p_ref) * log(p_new / p_ref))
}

📝 **Cálculo.** Arma `tb_drift`, con una fila por variable, y la graba como `drift_psi.txt` en la carpeta del experimento. La salida de esta celda es el conteo de variables por sugerencia.

In [ ]:
# variables a analizar: originales y new_, todas menos identificadores y clase
campos_drift <- setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria"))

# meses a comparar contra el de referencia
meses_drift <- sort(unique(c(PARAM$drift$meses_historia, PARAM$drift$mes_aplicacion)))
meses_comparar <- setdiff(meses_drift, PARAM$drift$mes_referencia)

# filas de cada mes
filas_ref <- dataset[, which(foto_mes == PARAM$drift$mes_referencia)]
filas_mes <- lapply(meses_comparar, function(m) dataset[, which(foto_mes == m)])

# una fila por variable, una columna de PSI por mes comparado
tb_drift <- rbindlist(lapply(campos_drift, function(campo) {
  x <- as.numeric(dataset[[campo]])
  valores <- sapply(filas_mes, function(f) psi(x[filas_ref], x[f], PARAM$drift$bins))
  names(valores) <- paste0("psi_", meses_comparar)
  as.data.table(c(list(variable= campo), as.list(round(valores, 4))))
}))

# resumen: maximo dentro de la historia, valor contra el mes de aplicacion y maximo general
cols_historia <- paste0("psi_", setdiff(PARAM$drift$meses_historia, PARAM$drift$mes_referencia))
tb_drift[, psi_max_historia := do.call(pmax, .SD), .SDcols= cols_historia]
tb_drift[, psi_aplicacion := get(paste0("psi_", PARAM$drift$mes_aplicacion))]
tb_drift[, psi_max := pmax(psi_max_historia, psi_aplicacion)]

tb_drift[, sugerencia := fcase(
  psi_max >= PARAM$drift$umbral_fuerte, "CAMBIO FUERTE: excluir o transformar",
  psi_max >= PARAM$drift$umbral_revisar, "REVISAR",
  default= "estable"
)]

setorder(tb_drift, -psi_max)

# queda grabada en la carpeta del experimento
fwrite(tb_drift, file= "drift_psi.txt", sep= "\t")

tb_drift[, .N, keyby= sugerencia]

📝 **Variables que más cambiaron.** Ordenadas de mayor a menor cambio. Columnas:

* `psi_202103`, `psi_202105`, `psi_202106`, `psi_202108`: PSI de cada mes contra 202104.
* `psi_max_historia`: el mayor entre 202103, 202105 y 202106. Es el que importa para el esquema de testeo (train en 202103 y 202104, test en 202106).
* `psi_aplicacion`: PSI contra 202108. Es el que importa para el submit final.
* `psi_max` y `sugerencia`: el mayor de los dos anteriores y su lectura según los umbrales.

In [ ]:
# variables que mas cambiaron su distribucion, de mayor a menor
tb_drift[sugerencia != "estable"]

📝 Las de cambio fuerte, en formato para copiar y pegar en `PARAM$drift$excluir`.

In [ ]:
# lista lista para copiar y pegar en PARAM$drift$excluir
dput(tb_drift[psi_max >= PARAM$drift$umbral_fuerte, variable])

📝 **Detalle de una variable.** `ver_drift("variable")` muestra mes a mes el porcentaje de `NA`, el porcentaje de ceros, la media, la mediana y el percentil 90. Sirve para ver qué tipo de cambio hubo:

* Si en un mes la variable se va a cero o a `NA`, está rota en ese mes: lo simple es excluirla.
* Si se corre de a poco todos los meses (típico de montos en pesos, por inflación), se puede transformar en lugar de excluir.

In [ ]:
# detalle de una variable mes a mes, para entender que tipo de cambio tuvo
ver_drift <- function(campo) {
  dataset[foto_mes %in% meses_drift, list(
    filas= .N,
    pct_na= round(100 * mean(is.na(get(campo))), 1),
    pct_cero= round(100 * mean(get(campo) == 0, na.rm= TRUE), 1),
    media= mean(as.numeric(get(campo)), na.rm= TRUE),
    mediana= quantile(as.numeric(get(campo)), 0.5, na.rm= TRUE, names= FALSE),
    p90= quantile(as.numeric(get(campo)), 0.9, na.rm= TRUE, names= FALSE)
  ), keyby= foto_mes]
}

# ejemplo: la variable que mas cambio
ver_drift(tb_drift[1, variable])

En esa exploracion gráfica del Data Drifting, se vio que estas variables cambian de distribucion entre 202104 y 202106, y se decidió brutalmente eliminarlas
* cprestamos_personales
* mprestamos_personales


más adelantes, se utilizarán técnicas más inclusivas, y se preservaran dichas variables, transformadas.


📝 **Decisión.** Lista de variables que quedan fuera del modelo. Por defecto están las dos que excluyó la cátedra; editala según lo que muestre la tabla. Se aplica en 4.2, al armar `campos_buenos`.

* Con `PARAM$drift$con_lags <- TRUE` también salen el `_lag1` y el `_delta1` de cada variable excluida.
* Las features `new_` calculadas con una variable excluida no salen solas. Tienen su propia fila en la tabla de drifting; además, en 4.2 se listan las que usan variables excluidas. Si querés sacarlas, las agregás a esta misma lista.

In [ ]:
# variables que se dejan afuera del modelo por data drifting
#  editar esta lista despues de mirar la tabla de arriba
PARAM$drift$excluir <- c("cprestamos_personales", "mprestamos_personales")

# TRUE: tambien salen el _lag1 y el _delta1 de cada variable excluida
PARAM$drift$con_lags <- TRUE

### 3.4 Lags y delta lags

📝 **Lags y delta lags.** Para todas las variables numéricas, originales y `new_`:

* `variable_lag1`: valor de la variable en el mes anterior, para el mismo cliente.
* `variable_delta1`: valor actual menos el del mes anterior.

Si el cliente no estaba en el mes anterior, ambos quedan en `NA`. Eso pasa en todas las filas de 202103, porque es el primer mes del dataset: la tabla de control de la celda lo muestra como 100% sin lag.

Esta celda además reordena `dataset` por cliente y mes, y al final se queda solo con los meses que se modelan (train y test), para ahorrar memoria. De acá en adelante `dataset` ya no tiene 202105, 202107 ni 202108, así que `ver_drift()` deja de servir.

In [ ]:
# variables a las que se les calcula lag1 y delta1:
#  todas las numericas, originales y new_, menos identificadores y clase
campos_lag <- setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria"))
campos_lag <- campos_lag[sapply(campos_lag, function(campo) is.numeric(dataset[[campo]]))]

# numero de mes consecutivo, para detectar huecos
dataset[, periodo0 := as.integer(foto_mes %/% 100) * 12L + as.integer(foto_mes %% 100)]

# para ahorrar memoria me quedo con los meses que se modelan y sus meses anteriores
meses_modelo <- unique(c(PARAM$train, PARAM$train_final, PARAM$future))
periodos_modelo <- as.integer(meses_modelo %/% 100) * 12L + as.integer(meses_modelo %% 100)
dataset <- dataset[periodo0 %in% c(periodos_modelo, periodos_modelo - 1L)]

# ordeno por cliente y mes: la fila anterior es el mes anterior del mismo cliente
setorder(dataset, numero_de_cliente, periodo0)

# lag1: valor de la fila anterior
cols_lag1 <- paste0(campos_lag, "_lag1")
dataset[, (cols_lag1) := shift(.SD, n= 1L, type= "lag"), .SDcols= campos_lag]

# filas sin mes anterior: primera fila, cambio de cliente o hueco de meses
#  en esas filas el lag queda en NA (pasa en todo 202103, que es el primer mes del dataset)
sin_lag <- dataset[, which(
  is.na(shift(periodo0)) |
  numero_de_cliente != shift(numero_de_cliente) |
  periodo0 - shift(periodo0) != 1L
)]
for (columna in cols_lag1) set(dataset, i= sin_lag, j= columna, value= NA)

# delta1: valor actual menos valor del mes anterior
for (campo in campos_lag) {
  set(dataset, j= paste0(campo, "_delta1"),
    value= dataset[[campo]] - dataset[[paste0(campo, "_lag1")]])
}

# control: porcentaje de filas sin mes anterior en cada mes
dataset[, con_lag := TRUE]
set(dataset, i= sin_lag, j= "con_lag", value= FALSE)
tb_lags <- dataset[foto_mes %in% meses_modelo,
  list(filas= .N, pct_sin_lag= round(100 * mean(!con_lag), 1)), keyby= foto_mes]

# me quedo solo con los meses que se modelan y borro las columnas auxiliares
dataset <- dataset[foto_mes %in% meses_modelo]
dataset[, c("periodo0", "con_lag") := NULL]
invisible(gc(full= TRUE, verbose= FALSE))

cat("Variables con lag y delta:", length(campos_lag), "\n")
cat("Columnas del dataset:", ncol(dataset), "\n")
tb_lags

## 4.0 Training y optimización de hiperparámetros

### 4.1 Training y validation

📝 **Armado de training y validation** (las cuatro celdas de código siguientes):

1. Se queda con los meses de `PARAM$train` (202103 y 202104).
2. Particiona 70/30 estratificado (`fold` 1 y 2).
3. Crea `clase01`: BAJA+1 y BAJA+2 valen 1, CONTINUA vale 0.
4. Marca `training = 1` para el fold 1 con undersampling (todos los BAJA + 10% de los CONTINUA).

Tener en cuenta que validation es todo lo que tiene `training = 0`: el fold 2 completo **más** los CONTINUA del fold 1 que no salieron sorteados. No hay filas compartidas con training, pero la proporción de BAJA en validation es menor que la real.

In [ ]:
# >> se queda con los meses de PARAM$train
dataset_train <- dataset[foto_mes %in% PARAM$train]

In [ ]:
  # >> agrega la columna fold: 1 = 70%, 2 = 30%, estratificado por clase_ternaria
  particionar(dataset_train,
    division= c(PARAM$training_pct, 100L -PARAM$training_pct),
    agrupa= "clase_ternaria",
    seed= PARAM$semilla_primigenia # aqui se usa SU semilla
  )

In [ ]:
# paso la clase a binaria que tome valores {0,1}  enteros
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0
#  a partir de ahora ya NO puedo cortar  por prob(BAJA+2) > 1/40

# >> al juntar BAJA+1 con BAJA+2 la probabilidad que predice el modelo es P(BAJA+1 o BAJA+2),
# >> por eso ya no vale el corte teorico 1/40 y despues se corta por cantidad de envios
dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [ ]:
# defino los datos que forma parte del training
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

# >> ojo: el comentario original dice SEGUNDA semilla, pero el codigo usa la primigenia
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
# >> numero al azar uniforme [0,1] por fila
dataset_train[, azar := runif(nrow(dataset_train))]
dataset_train[, training := 0L]

# >> training = 1 solo para filas del fold 1 que sean BAJA (todas) o CONTINUA con azar <= 0.1
# >> todo lo demas queda en training = 0 (fold 2 completo + los CONTINUA del fold 1 no sorteados)
dataset_train[
  foto_mes %in%  PARAM$train &  fold==1 &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),
  training := 1L
]

### 4.2 Variables predictoras

📝 **Variables predictoras.** `campos_buenos` arranca con todas las columnas y se le sacan, en este orden:

1. La clase y las columnas auxiliares del armado de training.
2. Las de `PARAM$campos_no_predictores` (`foto_mes`).
3. Las excluidas por drifting (`PARAM$drift$excluir`, definida en 3.3), con sus lags y deltas.

`numero_de_cliente` sigue entrando como predictora, igual que en el notebook de la cátedra. La segunda celda imprime qué se excluyó, qué features `new_` usan variables excluidas y cuántas predictoras quedan.

In [ ]:
# los campos que se van a utilizar
campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "fold", "azar", "training")
)

# saco las columnas que no son predictoras
campos_buenos <- setdiff(campos_buenos, PARAM$campos_no_predictores)

In [ ]:
# saco las variables excluidas por data drifting (lista definida en 3.3)
no_existen <- setdiff(PARAM$drift$excluir, colnames(dataset_train))
if (length(no_existen) > 0) stop("En PARAM$drift$excluir hay variables que no existen: ",
  paste(no_existen, collapse= ", "))

campos_excluidos <- PARAM$drift$excluir
if (PARAM$drift$con_lags) {
  campos_excluidos <- c(campos_excluidos,
    paste0(PARAM$drift$excluir, "_lag1"), paste0(PARAM$drift$excluir, "_delta1"))
}
campos_excluidos <- intersect(campos_excluidos, campos_buenos)
campos_buenos <- setdiff(campos_buenos, campos_excluidos)

# aviso: features new_ calculadas con variables excluidas, que siguen en el modelo
#  para sacarlas hay que agregarlas a PARAM$drift$excluir
new_afectadas <- names(insumos_new)[sapply(insumos_new, function(v) any(v %in% PARAM$drift$excluir))]
new_afectadas <- intersect(new_afectadas, campos_buenos)

cat("Excluidas por drifting:", length(campos_excluidos), "\n")
cat(" ", paste(campos_excluidos, collapse= ", "), "\n")
cat("Features new_ que usan variables excluidas y siguen en el modelo:", length(new_afectadas), "\n")
cat(" ", paste(new_afectadas, collapse= ", "), "\n")
cat("Variables predictoras:", length(campos_buenos), "\n")

### 4.3 Prueba rápida antes de la BO

📝 **Prueba rápida.** Mide si los cambios hechos más arriba (features, meses de training, exclusiones) mejoran o empeoran, sin esperar a la BO. Entrena un modelo por cada semilla de `PARAM$prueba$semillas` (5 por defecto), todos con los mismos hiperparámetros fijos y los mismos datos, y calcula la ganancia en 202106.

* **Hiperparámetros**: los mejores del experimento `PARAM$prueba$experimento_hp`. Al quedar fijos, la diferencia entre dos pruebas se debe a los datos y no a los hiperparámetros.
* **Datos de training**: las mismas filas que usa cada evaluación de la BO (fold 1 con undersampling), para que sea rápido.
* **Ganancia**: para cada cantidad de envíos se promedia la ganancia de las semillas, y sobre esa curva promedio se busca el máximo. En el gráfico, la línea azul es el promedio y las grises son las semillas.
* **Detalle por semilla**: la tabla muestra el máximo propio de cada semilla y su ganancia en la cantidad de envíos óptima del promedio, con el desvío entre semillas.
* **Historial**: cada prueba agrega una fila a `outputs/pruebas_rapidas_semillas.txt`, común a todos los experimentos, para comparar contra las anteriores.

Cómo leerla: sirve para comparar variantes entre sí, no como estimación de la ganancia final. El modelo final entrena con todas las filas y con hiperparámetros optimizados para este dataset. El desvío entre semillas da la escala del ruido: una diferencia entre dos pruebas menor que ese desvío no es concluyente. El promedio reduce el ruido propio de LightGBM, pero no el del sorteo de la partición y del undersampling, que dependen de la semilla primigenia y quedan fijos.

In [ ]:
# hiperparametros de la prueba: los mejores del experimento indicado, o los manuales
archivo_hp <- paste0("/content/buckets/b1/outputs/", PARAM$prueba$experimento_hp, "/BO_log.txt")

if (file.exists(archivo_hp)) {
  tb_hp <- fread(archivo_hp)
  setorder(tb_hp, -metrica)
  hp_prueba <- as.list(tb_hp[1, list(num_iterations, num_leaves, min_sum_hessian_in_leaf)])
  cat("Hiperparametros tomados de", archivo_hp, "\n")
} else {
  hp_prueba <- PARAM$prueba$hp_manual
  cat("No existe", archivo_hp, "\nSe usan los de PARAM$prueba$hp_manual\n")
}

param_prueba <- modifyList(PARAM$lgbm$param_fijos, hp_prueba)
unlist(hp_prueba)

In [ ]:
# entreno un modelo por semilla con las mismas filas que usa cada evaluacion de la BO:
#  fold 1 (70%) de los meses de train, con undersampling
t0_prueba <- Sys.time()

mtrain_prueba <- data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE])
label_prueba <- dataset_train[training == 1L, clase01]

# mes de testeo
dfuture_prueba <- dataset[foto_mes %in% PARAM$future]
mfuture_prueba <- data.matrix(dfuture_prueba[, campos_buenos, with= FALSE])

# por cada semilla: curva de ganancia acumulada e importancia de variables
curvas <- list()
importancias_prueba <- list()

for (semilla in PARAM$prueba$semillas) {
  param_semilla <- modifyList(param_prueba, list(seed= semilla))

  modelo <- lgb.train(
    data= lgb.Dataset(data= mtrain_prueba, label= label_prueba),
    params= param_semilla
  )

  prob <- predict(modelo, mfuture_prueba)
  curvas[[as.character(semilla)]] <- curva_ganancia(prob, dfuture_prueba$clase_ternaria)$ganancia_acumulada
  importancias_prueba[[as.character(semilla)]] <- lgb.importance(modelo)
  cat("semilla", semilla, "lista\n")
}

# una columna por semilla
curvas_prueba <- do.call(cbind, curvas)

# curva promedio de las semillas, para cada cantidad de envios
tb_prueba <- data.table(
  pos= seq_len(nrow(curvas_prueba)),
  ganancia_acumulada= rowMeans(curvas_prueba)
)

minutos_prueba <- round(as.numeric(difftime(Sys.time(), t0_prueba, units= "mins")), 1)
cat("Filas de training:", nrow(mtrain_prueba), " Variables:", length(campos_buenos),
  " Semillas:", ncol(curvas_prueba), " Minutos:", minutos_prueba, "\n")

rm(mtrain_prueba, mfuture_prueba, dfuture_prueba, curvas, modelo)
invisible(gc(full= TRUE, verbose= FALSE))

In [ ]:
# ganancia maxima de la curva promedio, cantidad de envios y grafico
mejor_prueba <- graficar_ganancia(tb_prueba,
  paste0("Prueba rapida ", PARAM$experimento, " - ganancia en ", paste(PARAM$future, collapse= ", ")),
  curvas_semillas= curvas_prueba)

In [ ]:
# detalle por semilla
#  ganancia_max y envios_max: el maximo propio de cada semilla
#  ganancia_en_corte: ganancia de esa semilla en la cantidad de envios optima del promedio
tb_semillas <- data.table(
  semilla= PARAM$prueba$semillas,
  ganancia_max= apply(curvas_prueba, 2, max),
  envios_max= apply(curvas_prueba, 2, which.max),
  ganancia_en_corte= curvas_prueba[mejor_prueba$pos, ]
)

desvio_prueba <- sd(tb_semillas$ganancia_en_corte)
cat("Desvio entre semillas en", mejor_prueba$pos, "envios:",
  format(round(desvio_prueba), big.mark= ".", decimal.mark= ","), "\n")

tb_semillas

In [ ]:
# agrego esta prueba al historial, que es comun a todos los experimentos
loguear(list(
  experimento= PARAM$experimento,
  train= paste(PARAM$train, collapse= "+"),
  future= paste(PARAM$future, collapse= "+"),
  variables= length(campos_buenos),
  num_iterations= hp_prueba$num_iterations,
  num_leaves= hp_prueba$num_leaves,
  min_sum_hessian_in_leaf= hp_prueba$min_sum_hessian_in_leaf,
  semillas= length(PARAM$prueba$semillas),
  ganancia_promedio= round(mejor_prueba$ganancia_acumulada),
  envios= mejor_prueba$pos,
  desvio= round(desvio_prueba),
  ganancia_min= min(tb_semillas$ganancia_en_corte),
  ganancia_max= max(tb_semillas$ganancia_en_corte),
  minutos= minutos_prueba
), arch= PARAM$prueba$archivo_log, verbose= FALSE)

# historial de pruebas rapidas, para comparar contra las anteriores
fread(PARAM$prueba$archivo_log, colClasses= list(character= "fecha"))

📝 **Importancia de variables.** Promedio de las semillas de la prueba rápida. La medida es el *gain*: qué proporción de la mejora total del modelo aporta cada variable. Dice cuánto la usa el modelo, no en qué dirección influye.

* **Tabla por tipo**: cuántas variables de cada tipo usa el modelo y qué porcentaje del gain total suman. Es la forma rápida de ver si los lags, los deltas y las features nuevas están aportando.
* **Gráfico**: las 30 variables más importantes, coloreadas por tipo.
* **Archivo**: la tabla completa queda en `importancia_prueba.txt`, en la carpeta del experimento.

In [ ]:
# importancia de variables, promedio de las semillas de la prueba rapida
tb_importancia_prueba <- resumir_importancia(importancias_prueba)
fwrite(tb_importancia_prueba, file= "importancia_prueba.txt", sep= "\t")

graficar_importancia(tb_importancia_prueba,
  paste0("Prueba rapida ", PARAM$experimento, " - variables mas importantes"))

### 4.4 Configuración de la BO

📝 **Contador de iteraciones.** `giter` numera las evaluaciones en el log. Si en la carpeta ya existe un `BO_log.txt` (corrida anterior con el mismo nombre de experimento), continúa la numeración y las filas nuevas se agregan al mismo archivo.

In [ ]:
# >> contador global de evaluaciones de la BO
giter <- 0

# >> si ya hay un BO_log.txt en la carpeta del experimento, sigue numerando desde ahi
# >> (solo continua la numeracion del log; la BO en si arranca de cero)
if( file.exists("BO_log.txt") ){
  tb_BO <- fread("BO_log.txt")
  giter <- tb_BO[, max(iter)]
}


📝 **Función objetivo de la BO.** Por cada combinación de hiperparámetros: entrena un LightGBM en training, mide `average_precision` en validation, graba la fila en `BO_log.txt` y devuelve la métrica.

La BO maximiza **average precision**, no ganancia (el comentario original "obtengo la ganancia" no refleja lo que hace el código).

In [ ]:
# En el argumento x llegan los parametros de la bayesiana

Estimar_lightgbm <- function(xs) {

  flush.console()
  giter <<- giter + 1

  # x pisa (o agrega) a param_fijos
  # >> combina los parametros fijos con los 3 que propone la BO
  param_completo <- modifyList(PARAM$lgbm$param_fijos, xs)
  xx <- copy(param_completo)

  # entreno LightGBM
  # >> entrena en dtrain y evalua average_precision en dvalidate en cada iteracion
  modelo <- lgb.train(
    data= dtrain,
    valids= list( "val"=dvalidate),
    param= param_completo
  )


  # obtengo la ganancia
  xx$iter <- giter
  # >> ojo: aca NO se calcula ganancia. best_score = mejor average_precision en validation
  # >> a lo largo de las iteraciones del boosting
  xx$metrica <- modelo$best_score

  # >> agrega una fila a BO_log.txt (queda en Drive, en la carpeta del experimento)
  loguear( xx, "BO_log.txt")
  set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")  # le reordeno a mlr3MBO

  # >> valor que la BO va a maximizar
  return( list( metrica = xx$metrica) )
}

📝 **Datasets en formato LightGBM.** Convierte training y validation a `lgb.Dataset` (matriz numérica + etiqueta `clase01`).

In [ ]:
# dejo los datos de training en el formato que necesita LightGBM

# >> training: filas con training == 1 (fold 1 con undersampling), solo campos_buenos
dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

In [ ]:
# datos de validatin en el formato que necesita LightGBM
# >> validation: TODAS las filas con training == 0, sin undersampling
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 0L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 0L, clase01],
  free_raw_data= FALSE
)

nrow(dvalidate)
ncol(dvalidate)

📝 **Configuración de la BO** con `bbotk` + `mlr3mbo`:

* **Objetivo**: `Estimar_lightgbm`, sobre el espacio de búsqueda definido antes, maximizando `metrica`.
* **Terminador**: corta al alcanzar `PARAM$hyperparametertuning$iteraciones` evaluaciones.
* **Surrogate**: Gaussian Process (`regr.km`, de DiceKriging).
* **Adquisición**: Expected Improvement, optimizada por búsqueda aleatoria de 1.000 puntos.
* **Loop**: `bayesopt_ego` (Efficient Global Optimization).

In [ ]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
# >> (esta variable no se usa mas abajo)
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- Estimar_lightgbm # la funcion que voy a maximizar

# >> funcion objetivo: recibe los hiperparametros (domain) y devuelve `metrica` a maximizar
objective <- ObjectiveRFun$new(
  fun= funcion_optimizar,
  domain= PARAM$hypeparametertuning$hs,
  codomain= ps(metrica = p_dbl(tags="maximize")),
  id= "BO_lightgbm"
)


# >> instancia de optimizacion: objetivo + criterio de corte por cantidad de evaluaciones
instance <- OptimInstanceBatchSingleCrit$new(
  objective= objective,
  terminator=  trm("evals", n_evals = PARAM$hyperparametertuning$iteraciones)
)


# >> surrogate: Gaussian Process (kriging) que aproxima metrica = f(hiperparametros)
# >> nugget.estim=TRUE: asume que la metrica observada tiene ruido
surrogate <- srlrn(lrn("regr.km",
  optim.method = "BFGS",
  nugget.estim = TRUE,
  control= list(trace = FALSE)))

# >> funcion de adquisicion: elige el proximo punto por Expected Improvement
acq_function  <- acqf("ei")                 # expected improvement

# >> el maximo de EI se busca probando 1000 puntos al azar sobre el surrogate
acq_optimizer  <- acqo(
  optimizer  = opt("random_search", batch_size= 1000),
  terminator = trm("evals", n_evals= 1000)
)

# >> arma el optimizador bayesiano (EGO) con las tres piezas anteriores
optimizer <- opt("mbo",
  loop_function = bayesopt_ego,
  surrogate= surrogate,
  acq_function= acq_function,
  acq_optimizer= acq_optimizer
)



### 4.5 Corrida de la BO

📝 **Corrida.** Ejecuta la optimización. Cada evaluación imprime su fila del log: las últimas 5 columnas son `num_iterations`, `num_leaves`, `min_sum_hessian_in_leaf`, `iter` y `metrica` (average precision en validation). Con dos meses de training y más de 500 variables, cada evaluación tarda bastante más que en el notebook 1.0.

In [ ]:
# >> corre la BO: es la celda mas lenta del notebook
bayesiana_salida <- optimizer$optimize(instance)

Luego de este punto, en la carpeta del experimento quedó el archivo BO_log.txt que posee todas las iteraciones de la Bayesian Optimization

## 5.0  Produccion

### 5.1 Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [ ]:
# >> CAMBIO: misma carpeta de salida en Drive (original: "/content/buckets/b1/exp")
setwd("/content/buckets/b1/outputs")
dir.create(PARAM$experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/outputs/", PARAM$experimento ))

📝 **Mejores hiperparámetros.** Lee `BO_log.txt` de la carpeta del experimento, lo ordena por métrica descendente y toma la primera fila. Si el log acumula corridas anteriores, las considera a todas.

In [ ]:
 # cergo los mejores hiperparametros, que son los de mayor metrica
 # >> lee el log de la BO y se queda con la fila de mayor metrica
 tb_bayesiana <- fread("BO_log.txt")
 setorder( tb_bayesiana, -metrica )
 PARAM$out$lgbm$mejores_hiperparametros <- as.list(tb_bayesiana[1])

#### 5.2 Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la  optimización bayesiana

📝 **Dataset de entrenamiento final.** A diferencia de la BO, acá se entrena con **todas** las filas de 202103 y 202104: sin partición 70/30 y sin undersampling. Los hiperparámetros se eligieron sobre un dataset mucho más chico que el que se usa en el modelo final.

In [ ]:
# clase01
# >> clase binaria ahora sobre todo `dataset` (meses de train y de test)
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
# >> pisa dataset_train: ahora son todas las filas de PARAM$train_final, sin particion y sin undersampling
dataset_train <- dataset[foto_mes %in% PARAM$train_final]
dataset_train[,.N, list(foto_mes, clase_ternaria)]

#### 5.3 Final Training Hyperparameters

In [ ]:
# >> parametros fijos + mejores hiperparametros de la BO
# >> (tambien entran fecha, iter y metrica del log, que no son parametros de LightGBM)
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

In [ ]:
# aplico el modelo a los datos sin clase
# >> 202106: mes de testeo. Tiene clase conocida, por eso se puede medir ganancia
dfuture <- dataset[foto_mes %in% PARAM$future]


📝 **Entrenamiento final y predicción.** Entrena un único LightGBM con los mejores hiperparámetros y predice la probabilidad de baja para cada cliente de 202106.

In [ ]:
# Entreno el modelo sobre todo el periodo

dtrain_final <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

# >> un unico modelo, sin validation: entrena las num_iterations elegidas
modelo_final <- lgb.train(
  data= dtrain_final,
  param= param_final
)

In [ ]:
# hago el predict en los datos del futuro, y calculo la ganancia

# aplico el modelo a los datos nuevos
# >> probabilidad de clase01 = 1 para cada cliente de 202106
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)



📝 **Ganancia.** Ordena los clientes de 202106 por probabilidad descendente y, para cada corte de `PARAM$cortes`, suma la ganancia de enviar la campaña a los primeros N. Cada BAJA+2 acertado suma 1.072.500 y cualquier otro envío resta 27.500 (los BAJA+1 se entrenaron como positivos, pero acá restan).

Referencia: la corrida de la cátedra (solo 202104, sin features nuevas ni lags) dio un máximo de **431.750.000 con 11.500 envíos**.

La tabla solo se imprime en pantalla: no se graba en Drive. Tampoco se guarda el modelo (`lgb.save` está comentado).

In [ ]:
# calculo la ganancia en los datos ddel futuro para todos los cortes
tb_prediccion <- dfuture[, list(numero_de_cliente, foto_mes,clase_ternaria)]
tb_prediccion[, prob := prediccion ]
# >> ganancia por envio: acierto BAJA+2 = 2.200.000 * 50% - 27.500 = 1.072.500
# >> cualquier otro (CONTINUA o BAJA+1) = -27.500
tb_prediccion[, gan := ifelse(clase_ternaria == "BAJA+2", 1072500, -27500)]

# lgb.save(modelo_final, "modelo.txt" )
# Dibujo la curva de ganancia acumulada
# >> ordena de mayor a menor probabilidad y acumula la ganancia
setorder(tb_prediccion, -prob)
tb_prediccion[, ganancia_acumulada := cumsum(gan)]
tb_prediccion[, pos := sequence(.N)]

# >> ganancia de enviar la campaña a los primeros `envios` clientes del ranking
# >> solo se imprime en pantalla: no se graba ningun archivo
for (envios in PARAM$cortes) {
  ganancia <- tb_prediccion[1:envios, sum(gan)]
  cat( envios, "\t", ganancia, "\n")
}

📝 **Ganancia máxima y curva.** Busca el máximo sobre toda la curva de ganancia acumulada, es decir, para cualquier cantidad de envíos y no solo para los cortes de `PARAM$cortes`, y dibuja la ganancia según la cantidad de envíos marcando ese máximo.

El máximo exacto se elige mirando la clase real de 202106, así que es un valor optimista. Lo que conviene mirar en el gráfico es qué tan ancha es la meseta alrededor del máximo: cuanto más plana, menos sensible es la ganancia a la cantidad de envíos que se elija.

In [ ]:
# ganancia maxima, cantidad de envios y curva del modelo final
mejor <- graficar_ganancia(tb_prediccion,
  paste0("Modelo final ", PARAM$experimento, " - ganancia en ", paste(PARAM$future, collapse= ", ")))

📝 **Importancia de variables del modelo final.** Misma lectura que en la prueba rápida, pero sobre el modelo final (una sola semilla). La tabla completa queda en `importancia_final.txt`.

In [ ]:
# importancia de variables del modelo final
tb_importancia <- resumir_importancia(list(lgb.importance(modelo_final)))
fwrite(tb_importancia, file= "importancia_final.txt", sep= "\t")

graficar_importancia(tb_importancia,
  paste0("Modelo final ", PARAM$experimento, " - variables mas importantes"))

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")